In [47]:
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import folium
import glob


from pathlib import Path

In [48]:
citibike_df = pd.read_csv("../data/JC/JC2025.csv")

In [49]:
citibike_df['started_at'] = pd.to_datetime(citibike_df['started_at'],errors="coerce")
citibike_df['ended_at'] = pd.to_datetime(citibike_df['ended_at'],errors="coerce")

In [50]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str')

In [51]:
missing_values = (
    citibike_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values["missing_share"] = (
    missing_values["missing_count"] / len(citibike_df)
)

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count,missing_share
7,end_station_id,4265,0.004480
10,end_lat,3425,0.003597
11,end_lng,3425,0.003597
6,end_station_name,3128,0.003285
4,start_station_name,3,0.000003
5,start_station_id,3,0.000003
8,start_lat,2,0.000002
9,start_lng,2,0.000002
0,ride_id,0,0.000000
1,rideable_type,0,0.000000


In [52]:
# ride duration 
citibike_df["ride_duration_minutes"] = (
    citibike_df["ended_at"] - citibike_df["started_at"]
).dt.total_seconds() / 60

In [53]:
#dropimg missing values and invalid ride durations 

citibike_df = citibike_df.dropna(
    subset=[
        "ride_id",
        "started_at",
        "ended_at",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng"
    ]
)

citibike_df = citibike_df[
    (citibike_df["ride_duration_minutes"] > 1) &
    (citibike_df["ride_duration_minutes"] <= 24 * 60)
].copy()

In [54]:
#time based variables

citibike_df["date"] = citibike_df["started_at"].dt.date
citibike_df["month"] = citibike_df["started_at"].dt.to_period("M").astype(str)
citibike_df["month_name"] = citibike_df["started_at"].dt.month_name()
citibike_df["day_of_week"] = citibike_df["started_at"].dt.day_name()
citibike_df["hour"] = citibike_df["started_at"].dt.hour

In [55]:
def assign_season(month_number):
    if month_number in [12, 1, 2]:
        return "Winter"
    elif month_number in [3, 4, 5]:
        return "Spring"
    elif month_number in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


citibike_df["season"] = (
    citibike_df["started_at"]
    .dt.month
    .apply(assign_season)
)

In [56]:
citibike_df[
    [
        "started_at",
        "date",
        "month",
        "month_name",
        "day_of_week",
        "hour",
        "season"
    ]
].head()

,started_at,date,month,month_name,day_of_week,hour,season
0,2025-11-18 18:34:14.943,2025-11-18,2025-11,November,Tuesday,18,Autumn
1,2025-11-26 16:29:15.513,2025-11-26,2025-11,November,Wednesday,16,Autumn
2,2025-11-04 22:31:58.010,2025-11-04,2025-11,November,Tuesday,22,Autumn
3,2025-11-08 06:51:57.424,2025-11-08,2025-11,November,Saturday,6,Autumn
4,2025-11-24 20:31:21.758,2025-11-24,2025-11,November,Monday,20,Autumn


In [57]:
citibike_df.to_csv("../data/JC/JC2025_Enriched.csv", index = False)